# Parse Inforcer Assessment PDFs into Delta Tables

**Default lakehouse:** ManagedServiceData

**Source folders:**
- `Files/copilot_readiness/` → copilot_readiness_assessments + _categories + _checks
- `Files/copilot_assessment/` → copilot_assessment_assessments only (summary reports)
- `Files/security_assessment/` → security_assessment_assessments + _categories + _checks

**Note:** Copilot Assessment PDFs are executive summaries and only contain overall scores/counts, not detailed category/check breakdowns.

**Features:**
- Recursively scans subfolders (supports date-based organization like `2026-06-13/`)
- Checkpoint tracking: skips already-processed PDFs
- Tables are upserted (MERGE / delete+insert) so re-runs are idempotent
- Requires: PyMuPDF
- Enhanced parsing to minimize NULL values
- Tenant names added to check rows for easier querying
- Business rationale extracted from check details

## How to run this notebook

1. Ensure the **ManagedServiceData** lakehouse is attached as the default lakehouse.
2. Upload assessment PDFs into the lakehouse `Files` folders:
   - `Files/copilot_readiness/` for Copilot Readiness assessments
   - `Files/copilot_assessment/` for Copilot Assessment reports
   - `Files/security_assessment/` for CIS Microsoft 365 security assessments
3. Run the cells in order:
   - `%pip install pymupdf --quiet`
   - The main ingestion cell (parsing + Delta upserts)
4. Use the validation cell at the bottom to inspect the populated Delta tables:
   - `copilot_readiness_assessments`, `copilot_readiness_categories`, `copilot_readiness_checks`
   - `copilot_assessment_assessments`, `copilot_assessment_categories`, `copilot_assessment_checks`
   - `security_assessment_assessments`, `security_assessment_categories`, `security_assessment_checks`

Re-running the notebook is safe: checkpoint tracking prevents reprocessing, tables are upserted/idempotent.

In [ ]:
%pip install pymupdf --quiet

In [ ]:
# Post-backfill verification: confirm business_rationale and tag fields are populated.
for tbl in ("copilot_readiness_checks", "security_assessment_checks"):
    df = spark.table(tbl)
    total = df.count()
    empty_rat = df.filter("business_rationale IS NULL OR business_rationale = ''").count()
    empty_name = df.filter("check_name IS NULL OR check_name = '' OR check_name = 'Unnamed Check'").count()
    populated_tag = df.filter("tags IS NOT NULL AND tags != ''").count()
    print(f"{tbl}: total={total}, empty_rationale={empty_rat} ({100*empty_rat/max(total,1):.1f}%), unnamed_checks={empty_name}, populated_tags={populated_tag}")

print("\nSample rows from security_assessment_checks:")
spark.sql("""
    SELECT check_name, status, priority, tags, LEFT(business_rationale, 120) AS rationale_preview
    FROM security_assessment_checks
    WHERE business_rationale IS NOT NULL AND business_rationale != ''
    LIMIT 5
""").show(truncate=False)

## Validation / Data preview

The cell below displays the Delta tables created by the ingestion logic. Use it to:

- Confirm that assessments, categories, and checks are populated.
- Spot-check a few rows for parsing quality (headers, scores, and check details).
- Verify that re-running the notebook updates existing assessments rather than duplicating them.

**Note:** Copilot Assessment tables only include _assessments (summary data). The _categories and _checks tables will only exist for Copilot Readiness and Security Assessment reports.

In [ ]:
# Fabric Notebook: Parse Inforcer Assessment PDFs into Delta Tables
#
# Default lakehouse must be set to: ManagedServiceData
#
# Reads PDFs from:
#   Files/copilot_readiness/*.pdf    -> copilot_readiness_assessments + _categories + _checks
#   Files/copilot_assessment/*.pdf   -> copilot_assessment_assessments (summary only)
#   Files/security_assessment/*.pdf  -> security_assessment_assessments + _categories + _checks
#
# Features:
#   - Checkpoint tracking: skips already-processed PDFs
#   - Tables are upserted (MERGE / delete+insert) so re-runs are idempotent
#   - Enhanced parsing to minimize NULL values with intelligent fallbacks
#   - Recursively scans date-based subfolders (e.g., 2026-06-15/)
#
# Requires: PyMuPDF  (install via %pip install pymupdf)

import re
import hashlib
from datetime import datetime, timezone, date
import fitz  # PyMuPDF
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
import notebookutils

# Base folders for assessment PDFs
COPILOT_READINESS_FOLDER = "Files/copilot_readiness"
COPILOT_ASSESSMENT_FOLDER = "Files/copilot_assessment"
SECURITY_ASSESSMENT_FOLDER = "Files/security_assessment"

# Legacy aliases for backward compatibility
COPILOT_BASE_FOLDER = COPILOT_READINESS_FOLDER
SECURITY_BASE_FOLDER = SECURITY_ASSESSMENT_FOLDER

VALID_STATUSES = {"Passed", "Failed", "Warning"}
VALID_PRIORITIES = {"High", "Medium", "Low"}
# Icon glyphs that anchor a check row in the PDF layout
ICON_CHARS = {"✕", "✓", "⚠", "✗", "!"}

# Checkpoint table name
CHECKPOINT_TABLE = "ingestion_checkpoints"

def get_dated_folder(base_folder, ingestion_date=None):
    """Generate a date-based folder path for organizing ingested files."""
    if ingestion_date is None:
        ingestion_date = date.today()
    date_str = ingestion_date.strftime('%Y-%m-%d')
    return f"{base_folder.rstrip('/')}/{date_str}"

def ensure_folder_exists(folder_path):
    """Validate folder path; folders are auto-created on first write in Fabric."""
    try:
        notebookutils.fs.ls(folder_path)
        print(f"✓ Folder exists: {folder_path}")
    except:
        print(f"ℹ️ Folder will be created on first write: {folder_path}")
    return folder_path

# --- Checkpoint tracking functions --------------------------------------------

def init_checkpoint_table():
    """Create checkpoint table if it doesn't exist."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CHECKPOINT_TABLE} (
            file_path STRING,
            file_size BIGINT,
            file_hash STRING,
            assessment_id STRING,
            table_prefix STRING,
            processed_at TIMESTAMP,
            status STRING,
            error_message STRING
        ) USING DELTA
    """)
    print(f"✓ Checkpoint table initialized: {CHECKPOINT_TABLE}")

def get_file_hash(file_path):
    """Calculate SHA256 hash of file for change detection (first 1MB)."""
    try:
        content = notebookutils.fs.head(file_path, 1024 * 1024)
        if isinstance(content, str):
            content = content.encode("ISO-8859-1", errors="ignore")
        return hashlib.sha256(content).hexdigest()[:32]
    except Exception as e:
        print(f"Warning: Could not hash {file_path}: {e}")
        return None

def is_file_processed(file_path, file_size, processed_cache=None):
    """Check if file has already been processed successfully (uses in-memory cache when provided)."""
    if processed_cache is not None:
        entry = processed_cache.get(file_path)
        if entry is None:
            return False
        if entry == file_size:
            return True
        print(f"  ℹ️ File size changed, reprocessing: {file_path}")
        return False
    try:
        result = spark.sql(f"""
            SELECT file_path, file_size, file_hash, status 
            FROM {CHECKPOINT_TABLE}
            WHERE file_path = '{file_path}'
            AND status = 'success'
        """).collect()
        if result:
            checkpoint = result[0]
            if checkpoint.file_size == file_size:
                return True
            print(f"  ℹ️ File size changed, reprocessing: {file_path}")
            return False
        return False
    except:
        return False

def load_processed_cache():
    """Load all successful checkpoints into a dict {file_path: file_size}."""
    try:
        rows = spark.sql(f"""
            SELECT file_path, file_size FROM {CHECKPOINT_TABLE}
            WHERE status = 'success'
        """).collect()
        return {r.file_path: r.file_size for r in rows}
    except Exception:
        return {}

def record_checkpoint(file_path, file_size, file_hash, assessment_id, table_prefix, status, error_message=None):
    """Record processing checkpoint for a file."""
    now = datetime.now(timezone.utc)
    # Explicit schema to avoid CANNOT_DETERMINE_TYPE when error_message is None
    checkpoint_schema = StructType([
        StructField("file_path", StringType(), True),
        StructField("file_size", LongType(), True),
        StructField("file_hash", StringType(), True),
        StructField("assessment_id", StringType(), True),
        StructField("table_prefix", StringType(), True),
        StructField("processed_at", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
    ])
    checkpoint_df = spark.createDataFrame(
        [(file_path, int(file_size) if file_size is not None else None, file_hash,
          assessment_id, table_prefix, now, status, error_message)],
        schema=checkpoint_schema,
    )
    checkpoint_df.createOrReplaceTempView("checkpoint_staging")
    spark.sql(f"""
        MERGE INTO {CHECKPOINT_TABLE} t 
        USING checkpoint_staging s 
        ON t.file_path = s.file_path
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

# --- IO helpers ---------------------------------------------------------------

def list_pdfs(folder_path):
    """Recursively list all PDF files in a folder and its subfolders."""
    pdf_files = []
    try:
        entries = notebookutils.fs.ls(folder_path)
    except Exception as e:
        print(f"Folder {folder_path} not accessible: {e}")
        return []
    for entry in entries:
        is_directory = entry.isDir if hasattr(entry, 'isDir') else entry.name.endswith('/')
        if is_directory:
            subfolder_path = f"{folder_path.rstrip('/')}/{entry.name.rstrip('/')}"
            pdf_files.extend(list_pdfs(subfolder_path))
        elif entry.name.lower().endswith('.pdf'):
            pdf_files.append(entry)
    return pdf_files


def read_pdf(file_path):
    """Open PDF and return (full_text, doc). Caller must close doc."""
    try:
        raw = spark.read.format("binaryFile").load(file_path).first()['content']
    except:
        raw = notebookutils.fs.head(file_path, 100 * 1024 * 1024)
        if isinstance(raw, str):
            raw = raw.encode("ISO-8859-1", errors="ignore")
    doc = fitz.open(stream=raw, filetype="pdf")
    text = "\n".join(page.get_text() for page in doc)
    return text, doc


def read_pdf_text(file_path):
    text, doc = read_pdf(file_path)
    doc.close()
    return text


def assessment_id_for(file_path):
    return hashlib.sha256(file_path.encode()).hexdigest()[:16]


# --- Shared parsers (both report types use the same Inforcer layout) ----------

def parse_header(text):
    """Parse assessment metadata from PDF header, using fallbacks to avoid NULLs."""
    out = {
        "assessment_name": None,
        "tenant_name": None,
        "assessment_date": None,
        "assessment_time": None,
    }
    patterns = [
        r"Assessment:\s*\n?\s*([^\n]+)",
        r"Assessment:\s+([A-Za-z ]+)",
        r"Assessment Report\s*[:\-]?\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            name = m.group(1).strip()
            if name.lower() not in {"tenant", "date", "time"}:
                out["assessment_name"] = name
                break
    patterns = [
        r"Tenant Assessment:\s*\n?\s*([^\n]+)",
        r"Tenant:\s*\n?Tenant Assessment:\s*([^\n]+)",
        r"Tenant:\s*\n?\s*([^\n]+)",
        r"Organization:\s*\n?\s*([^\n]+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            name = m.group(1).strip()
            if name and not name.startswith(("Assessment", "Date:", "Time:")):
                out["tenant_name"] = name
                break
    patterns = [
        r"Assessment Date:\s*\n?\s*([0-9]{4}-[0-9]{2}-[0-9]{2})",
        r"Assessment Date:\s*\n?\s*([0-9]{2}/[0-9]{2}/[0-9]{4})",
        r"Date:\s*\n?\s*([0-9]{4}-[0-9]{2}-[0-9]{2})",
        r"Date:\s*\n?\s*([0-9]{2}/[0-9]{2}/[0-9]{4})",
    ]
    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            out["assessment_date"] = m.group(1).strip()
            break
    if not out["assessment_date"]:
        out["assessment_date"] = date.today().strftime('%Y-%m-%d')
    patterns = [
        r"Assessment Time:\s*\n?\s*([0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9:.\-Z]+)",
        r"Time:\s*\n?\s*([0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9:.\-Z]+)",
        r"Assessment Time:\s*\n?\s*([0-9:.\-Z]+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            out["assessment_time"] = m.group(1).strip()
            break
    if not out["assessment_time"]:
        out["assessment_time"] = datetime.now(timezone.utc).isoformat()
    return out


def parse_executive_summary(text):
    """Parse executive summary scores. Returns 0 instead of NULL for missing values."""
    out = {"overall_score_pct": 0, "passed": 0, "failed": 0, "warnings": 0}
    m = re.search(r"Overall Score\s*\n\s*(\d+)\s*%", text)
    if m:
        out["overall_score_pct"] = int(m.group(1))
    else:
        m = re.search(r"Score:\s*(\d+)\s*%", text)
        if m:
            out["overall_score_pct"] = int(m.group(1))
    m = re.search(r"Passed\s*\n\s*(\d+)\s*\n", text)
    if m:
        out["passed"] = int(m.group(1))
    else:
        m = re.search(r"Passed:\s*(\d+)", text)
        if m:
            out["passed"] = int(m.group(1))
    m = re.search(r"Failed\s*\n\s*(\d+)\s*\n", text)
    if m:
        out["failed"] = int(m.group(1))
    else:
        m = re.search(r"Failed:\s*(\d+)", text)
        if m:
            out["failed"] = int(m.group(1))
    m = re.search(r"Warnings\s*\n\s*(\d+)", text)
    if m:
        out["warnings"] = int(m.group(1))
    else:
        m = re.search(r"Warning[s]?:\s*(\d+)", text)
        if m:
            out["warnings"] = int(m.group(1))
    return out


def parse_categories(text):
    """Parse categories from 'Assessment by Category (Top 5)' section."""
    results = []
    pattern = re.compile(
        r"([A-Za-z0-9 &\-]+?)\s+(\d+)\s*%\s*\n\s*(\d+)\s+passed\s+(\d+)\s+failed",
        re.IGNORECASE,
    )
    for m in pattern.finditer(text):
        name = m.group(1).strip()
        if name.lower() in {"overall score", "score"}:
            continue
        results.append({
            "category_name": name,
            "score_pct": int(m.group(2)),
            "passed": int(m.group(3)),
            "failed": int(m.group(4)),
        })
    if not results:
        pattern2 = re.compile(
            r"([A-Za-z0-9 &\-]+?)\s+(\d+)%\s+(\d+)\s+passed\s+(\d+)\s+failed",
            re.IGNORECASE,
        )
        for m in pattern2.finditer(text):
            name = m.group(1).strip()
            if name.lower() in {"overall score", "score"}:
                continue
            results.append({
                "category_name": name,
                "score_pct": int(m.group(2)),
                "passed": int(m.group(3)),
                "failed": int(m.group(4)),
            })
    return results


# --- Check-row parser (layout-aware) -----------------------------------------

CATEGORY_HEADER_RE = re.compile(
    r"^([A-Z][A-Za-z0-9 &/]+?)\s*\(([^)]+)\)\s*\((\d+)\s*checks?\)\s*$"
)

def parse_checks_layout(doc):
    """
    Parse checks from a PDF document using positional block extraction.
    Each check renders as: an anchor block (icon + name) in the left column,
    an optional tag block (left col), rationale block(s) (middle col), and a
    status/priority/framework block (right col). Reading order on a page is
    determined by anchor y-coordinates.
    """
    checks = []
    current_category = None
    current_subcategory = None
    for page_idx in range(len(doc)):
        page = doc[page_idx]
        blocks = [b for b in page.get_text("blocks") if b[6] == 0]
        blocks.sort(key=lambda b: (b[1], b[0]))

        anchors = []
        for b in blocks:
            x0, _, _, _, txt, *_ = b
            first = txt.strip().split("\n", 1)[0].strip()
            if x0 < 150 and first in ICON_CHARS:
                anchors.append(b)

        if not anchors:
            for b in blocks:
                m = CATEGORY_HEADER_RE.match(b[4].strip())
                if m:
                    current_category = m.group(1).strip()
                    current_subcategory = m.group(2).strip()
            continue

        for anc_i, anchor in enumerate(anchors):
            ax0, ay0, _, _, atxt, *_ = anchor
            y_end = anchors[anc_i + 1][1] if anc_i + 1 < len(anchors) else float("inf")

            for b in blocks:
                if b[1] < ay0:
                    m = CATEGORY_HEADER_RE.match(b[4].strip())
                    if m:
                        current_category = m.group(1).strip()
                        current_subcategory = m.group(2).strip()

            anchor_lines = [l.strip() for l in atxt.split("\n") if l.strip()]
            name_lines = anchor_lines[1:] if anchor_lines and anchor_lines[0] in ICON_CHARS else anchor_lines
            check_name = " ".join(name_lines).strip()

            tag_parts = []
            rationale_parts = []
            right_blocks = []

            for b in blocks:
                bx0, by0, _, _, btxt, *_ = b
                if b is anchor:
                    continue
                if by0 < ay0 - 5 or by0 >= y_end - 5:
                    continue
                txt_clean = btxt.strip()
                if not txt_clean:
                    continue
                if bx0 < 150:
                    tag_parts.append(txt_clean.replace("\n", " "))
                elif bx0 < 340:
                    rationale_parts.append((by0, txt_clean.replace("\n", " ")))
                else:
                    right_blocks.append((by0, txt_clean))

            tag = " ".join(tag_parts).strip()
            rationale_parts.sort(key=lambda p: p[0])
            rationale = " ".join(p[1] for p in rationale_parts).strip()

            status_word = None
            priority_word = None
            framework_lines_out = []
            if right_blocks:
                right_blocks.sort(key=lambda p: p[0])
                right_lines = []
                for _, t in right_blocks:
                    right_lines.extend(l.strip() for l in t.split("\n") if l.strip())
                seen_s = seen_p = False
                for ln in right_lines:
                    if not seen_s and ln in VALID_STATUSES:
                        status_word = ln
                        seen_s = True
                        continue
                    if seen_s and not seen_p and ln in VALID_PRIORITIES:
                        priority_word = ln
                        seen_p = True
                        continue
                    if seen_p:
                        framework_lines_out.append(ln)
            framework_raw = " | ".join(framework_lines_out)

            control_m = re.search(r"Control:\s*([0-9.]+)", framework_raw) if framework_raw else None
            level_m = re.search(r"Level:\s*\(?(L[12])\)?", framework_raw) if framework_raw else None

            checks.append({
                "category": current_category,
                "subcategory": current_subcategory,
                "check_name": check_name,
                "tag": tag,
                "business_rationale": rationale,
                "status": status_word,
                "priority": priority_word,
                "framework_raw": framework_raw,
                "control": control_m.group(1) if control_m else None,
                "level": level_m.group(1) if level_m else None,
            })

    return checks


# --- Build rows + upsert ------------------------------------------------------

def build_rows(file_info, text, doc, ingested_at, assessment_type="Security Assessment"):
    """Build assessment, category, and check rows from PDF text + doc."""
    aid = assessment_id_for(file_info.path)
    header = parse_header(text)
    summary = parse_executive_summary(text)
    categories = parse_categories(text)
    parsed_checks = parse_checks_layout(doc)

    assessment_name = header["assessment_name"] or f"Assessment from {file_info.name}"

    tenant_name = header["tenant_name"]
    if not tenant_name:
        filename = file_info.name
        if "Summary Copilot Assessment" in filename or "Copilot Assessment" in filename:
            tenant_from_file = filename.replace("Summary Copilot Assessment", "").replace("Copilot Assessment", "")
            tenant_from_file = tenant_from_file.replace(".pdf", "").strip()
            if tenant_from_file:
                tenant_name = tenant_from_file
        elif "Assessment_Copilot" in filename:
            parts = filename.split("_")
            if len(parts) >= 3:
                tenant_from_file = parts[-1].replace(".pdf", "").strip()
                if tenant_from_file:
                    tenant_name = tenant_from_file
    tenant_name = tenant_name or "Unknown Tenant"

    assessment_date = header["assessment_date"] or date.today().strftime('%Y-%m-%d')
    assessment_time = header["assessment_time"] or datetime.now(timezone.utc).isoformat()

    assessment_row = Row(
        assessment_id=aid,
        file_name=file_info.name,
        file_path=file_info.path,
        assessment_type=assessment_type,
        assessment_name=assessment_name,
        tenant_name=tenant_name,
        tenant_assessment_name=tenant_name,
        assessment_date=assessment_date,
        assessment_time=assessment_time,
        overall_score_pct=summary["overall_score_pct"],
        passed_count=summary["passed"],
        failed_count=summary["failed"],
        warnings_count=summary["warnings"],
        created_at=ingested_at,
        updated_at=ingested_at,
        ingested_at=ingested_at,
    )

    category_rows = [
        Row(
            assessment_id=aid,
            category_name=c["category_name"],
            score_pct=c["score_pct"],
            passed_count=c["passed"],
            failed_count=c["failed"],
            ingested_at=ingested_at,
        )
        for c in categories
    ]

    check_rows = []
    for c in parsed_checks:
        check_name = c.get("check_name") or "Unnamed Check"
        check_id = hashlib.sha256(f"{aid}_{check_name}".encode()).hexdigest()[:16]
        business_rationale = c.get("business_rationale", "") or ""
        status = c.get("status") or "Unknown"
        priority = c.get("priority") or "Medium"
        framework_raw = c.get("framework_raw") or ""
        frameworks = framework_raw or f"{assessment_type} Framework"
        category_label = c.get("category") or ""
        area_code = c.get("subcategory") or ""

        check_rows.append(Row(
            check_id=check_id,
            assessment_id=aid,
            tenant_name=tenant_name,
            category=category_label,
            subcategory=area_code,
            category_label=category_label,
            area_code=area_code,
            check_name=check_name,
            business_rationale=business_rationale,
            status=status,
            priority=priority,
            tags=c.get("tag", "") or "",
            frameworks=frameworks,
            framework_raw=framework_raw,
            framework_control=c.get("control") or "",
            framework_level=c.get("level") or "",
            control=c.get("control") or "",
            level=c.get("level") or "",
            created_at=ingested_at,
            updated_at=ingested_at,
            ingested_at=ingested_at,
        ))

    return assessment_row, category_rows, check_rows


def upsert(df, table_name, key_cols):
    if df.rdd.isEmpty():
        print(f"  no rows for {table_name}")
        return
    df.createOrReplaceTempView("staging")
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA AS SELECT * FROM staging WHERE 1=0")

    if len(key_cols) == 1:
        spark.sql(f"""
            MERGE INTO {table_name} t USING staging s ON t.{key_cols[0]} = s.{key_cols[0]}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
    else:
        ids = [r.assessment_id for r in df.select("assessment_id").distinct().collect()]
        id_list = ",".join([f"'{i}'" for i in ids])
        spark.sql(f"DELETE FROM {table_name} WHERE assessment_id IN ({id_list})")
        df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"  wrote {df.count()} row(s) to {table_name}")


def ingest_folder(folder_path, table_prefix, assessment_type="Security Assessment"):
    """Ingest all PDFs from a folder into Delta tables."""
    files = list_pdfs(folder_path)
    print(f"{folder_path}: {len(files)} PDF(s)")

    if not files:
        return {"total": 0, "processed": 0, "skipped": 0, "failed": 0}

    processed_cache = load_processed_cache()
    now = datetime.now(timezone.utc)
    all_assessments, all_categories, all_checks = [], [], []
    stats = {"total": len(files), "processed": 0, "skipped": 0, "failed": 0}

    for f in files:
        try:
            if is_file_processed(f.path, f.size, processed_cache):
                print(f"  ⏭️  Skipped (already processed): {f.name}")
                stats["skipped"] += 1
                continue

            file_hash = get_file_hash(f.path)
            text, doc = read_pdf(f.path)
            try:
                a, cats, chks = build_rows(f, text, doc, now, assessment_type)
            finally:
                doc.close()

            all_assessments.append(a)
            all_categories.extend(cats)
            all_checks.extend(chks)

            record_checkpoint(f.path, f.size, file_hash, a.assessment_id, table_prefix, "success")
            stats["processed"] += 1
            print(f"  ✅ Parsed {f.name}: {len(cats)} categories, {len(chks)} checks")

        except Exception as e:
            file_hash = get_file_hash(f.path)
            aid = assessment_id_for(f.path)
            record_checkpoint(f.path, f.size, file_hash, aid, table_prefix, "failed", str(e))
            stats["failed"] += 1
            print(f"  ❌ FAILED {f.name}: {e}")

    if all_assessments:
        upsert(spark.createDataFrame(all_assessments), f"{table_prefix}_assessments", ["assessment_id"])
    if all_categories:
        upsert(spark.createDataFrame(all_categories), f"{table_prefix}_categories", ["assessment_id", "category_name"])
    if all_checks:
        upsert(spark.createDataFrame(all_checks), f"{table_prefix}_checks", ["assessment_id", "check_name"])

    print(f"  📊 Stats: {stats['processed']} processed, {stats['skipped']} skipped, {stats['failed']} failed")
    return stats


print("✅ All functions loaded successfully!")

# --- Run ingestion (NON-DESTRUCTIVE - tables are upserted, not dropped) ------
print("Initializing checkpoint tracking...")
init_checkpoint_table()

copilot_readiness_folder = COPILOT_READINESS_FOLDER
copilot_assessment_folder = COPILOT_ASSESSMENT_FOLDER
security_folder = SECURITY_ASSESSMENT_FOLDER

print(f"\n📂 Processing all PDFs in base folders:")
print(f"📂 Copilot Readiness: {copilot_readiness_folder}")
print(f"📂 Copilot Assessment: {copilot_assessment_folder}")
print(f"📂 Security Assessment: {security_folder}\n")

print("\n" + "="*60)
print("🔄 Starting PDF Ingestion")
print("="*60 + "\n")

stats_readiness = ingest_folder(copilot_readiness_folder, "copilot_readiness", "Copilot Readiness")
stats_assessment = ingest_folder(copilot_assessment_folder, "copilot_assessment", "Copilot Assessment")
stats_security = ingest_folder(security_folder, "security_assessment", "Security Assessment")

print("\n" + "="*60)
print("📊 Overall Ingestion Summary")
print("="*60)
print(f"Copilot Readiness:  {stats_readiness['processed']} processed, {stats_readiness['skipped']} skipped, {stats_readiness['failed']} failed")
print(f"Copilot Assessment: {stats_assessment['processed']} processed, {stats_assessment['skipped']} skipped, {stats_assessment['failed']} failed")
print(f"Security Assessment: {stats_security['processed']} processed, {stats_security['skipped']} skipped, {stats_security['failed']} failed")
print("="*60)

total_processed = stats_readiness['processed'] + stats_assessment['processed'] + stats_security['processed']
total_skipped = stats_readiness['skipped'] + stats_assessment['skipped'] + stats_security['skipped']
total_failed = stats_readiness['failed'] + stats_assessment['failed'] + stats_security['failed']

print(f"\n✅ Total: {total_processed} processed, {total_skipped} skipped, {total_failed} failed")
print("="*60 + "\n")

if total_failed > 0:
    print("\n⚠️ Some files failed to process. Check checkpoint table for details:")
    print(f"   SELECT * FROM {CHECKPOINT_TABLE} WHERE status = 'failed'")

In [ ]:
# --- Preview tables and row counts (safe to re-run) -------------------------
print("="*60)
print("📊 Assessment Tables Preview")
print("="*60 + "\n")

print("🔵 Copilot Readiness Assessment Tables:")
display(spark.table("copilot_readiness_assessments"))
display(spark.table("copilot_readiness_categories"))
display(spark.table("copilot_readiness_checks"))

print("\n🔵 Copilot Assessment Tables (Summary Only):")
display(spark.table("copilot_assessment_assessments"))

print("\n🔵 Security Assessment Tables:")
display(spark.table("security_assessment_assessments"))
display(spark.table("security_assessment_categories"))
display(spark.table("security_assessment_checks"))

print("\n" + "="*60)
print("📈 Table Row Counts")
print("="*60)

tables = [
    "copilot_readiness_assessments",
    "copilot_readiness_categories",
    "copilot_readiness_checks",
    "copilot_assessment_assessments",
    "security_assessment_assessments",
    "security_assessment_categories",
    "security_assessment_checks",
]

for table in tables:
    try:
        count = spark.table(table).count()
        print(f"✓ {table}: {count} rows")
    except Exception:
        print(f"ℹ️  {table}: Table not found")

print("\n" + "="*60)
print("📋 Ingestion Checkpoint Status")
print("="*60)

try:
    checkpoint_summary = spark.sql(f"""
        SELECT 
            table_prefix,
            status,
            COUNT(*) as file_count,
            MAX(processed_at) as last_processed
        FROM {CHECKPOINT_TABLE}
        GROUP BY table_prefix, status
        ORDER BY table_prefix, status
    """)
    display(checkpoint_summary)
except Exception:
    print("Checkpoint table not found or empty")